# 多模态梯度冲突分析（数据组成自适应版）

本版专门面向两类数据构成场景：

1. **text-only + text-img 混合**
2. **仅 text-img**

核心优化：
- 数据处理按组成自动分流（composition-aware）
- 统计指标做 token 归一化，避免样本长度偏置
- 实验分析提供两条路径（混合数据 vs 纯视觉多模态数据）

In [ ]:
import ast, json, hashlib
from pathlib import Path
from typing import Optional, Iterable

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")


In [ ]:
LOG_PATH = Path("gradient_logs.jsonl")
OUT = Path("analysis_outputs")
CACHE = OUT / "vector_cache"
OUT.mkdir(exist_ok=True, parents=True)
CACHE.mkdir(exist_ok=True, parents=True)

STEP_MIN=None
STEP_MAX=None
STEP_MOD=None
MAX_ROWS=200000
MAX_VEC_ROWS=60000
MAX_PAIR=30000

assert LOG_PATH.exists(), LOG_PATH
print(LOG_PATH.resolve())


In [ ]:
def _step_ok(v, lo=None, hi=None, mod=None):
    try: s=int(v)
    except: return False
    if lo is not None and s<lo: return False
    if hi is not None and s>hi: return False
    if mod is not None and mod>1 and s%mod!=0: return False
    return True

def load_subset(path, usecols=None, partition_in=None, max_rows=None):
    usecols=set(usecols) if usecols else None
    rows=[]
    with path.open('r',encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not _step_ok(o['step'], STEP_MIN, STEP_MAX, STEP_MOD):
                continue
            if partition_in is not None and o.get('grad_partition','all') not in partition_in:
                continue
            rows.append(o if usecols is None else {k:o.get(k,None) for k in usecols})
            if max_rows and len(rows)>=max_rows: break
    df=pd.DataFrame(rows)
    if len(df)==0: return df
    for c in ['step','layer_depth','grad_norm','supervised_token_count']:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
    if 'grad_partition' in df.columns: df['grad_partition']=df['grad_partition'].fillna('all')
    if 'grad_was_none' in df.columns: df['grad_was_none']=df['grad_was_none'].fillna(False).astype(bool)
    return df


## 1) 数据组成识别（新增）

基于日志中的 `batch_mix_type / batch_*_examples`（如果存在）识别当前实验属于：
- 混合（text_and_vision）
- 纯 text-img/vision（vision_only）

In [ ]:
meta = load_subset(
    LOG_PATH,
    usecols=[
        'step','grad_partition','layer_depth','grad_norm','supervised_token_count','grad_was_none',
        'batch_mix_type','batch_has_text_only','batch_has_vision','batch_text_only_examples','batch_image_examples','batch_video_examples'
    ],
    max_rows=MAX_ROWS,
)

if len(meta)==0:
    raise ValueError('No rows loaded. check STEP_* filters')

for c in ['batch_mix_type','batch_has_text_only','batch_has_vision','batch_text_only_examples','batch_image_examples','batch_video_examples']:
    if c not in meta.columns:
        meta[c] = np.nan

# token归一化指标（避免长度偏置）
meta['grad_norm_per_token'] = meta['grad_norm'] / (meta['supervised_token_count'].clip(lower=1))

mix_dist = meta['batch_mix_type'].value_counts(dropna=False)
display(mix_dist)
meta.head(3)


In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(data=meta, x='grad_partition', order=[p for p in ['all','text_only','image_only'] if p in meta['grad_partition'].unique()])
plt.title('partition coverage')
plt.tight_layout(); plt.savefig(OUT/'fig_partition_coverage.png', dpi=180); plt.show()


## 2) 统一基础分析（token归一化）

In [ ]:
trend = meta.groupby(['step','grad_partition'],as_index=False)['grad_norm_per_token'].mean()
plt.figure(figsize=(12,5))
sns.lineplot(data=trend,x='step',y='grad_norm_per_token',hue='grad_partition')
plt.title('mean grad_norm_per_token over steps')
plt.tight_layout(); plt.savefig(OUT/'fig_trend_grad_per_token.png',dpi=180); plt.show()

lp = meta.groupby(['layer_depth','grad_partition'],as_index=False)['grad_norm_per_token'].mean()
piv = lp.pivot(index='layer_depth',columns='grad_partition',values='grad_norm_per_token').sort_index()
plt.figure(figsize=(10,8))
sns.heatmap(piv,cmap='magma')
plt.title('layer x partition (grad_norm_per_token)')
plt.tight_layout(); plt.savefig(OUT/'fig_layer_partition_grad_per_token.png',dpi=200); plt.show()


## 3) 场景A：text-only + text-img 混合数据优化分析

重点看：
- batch内 text-only 样本占比变化是否影响 image/text partition 强弱
- 是否出现某些层在 mixed batch 中竞争更剧烈

In [ ]:
if meta['batch_mix_type'].notna().any():
    m = meta[meta['batch_mix_type']=='text_and_vision'].copy()
    if len(m)>0:
        # 分区主导指数（token归一化版）
        key=['step','layer_depth']
        w=m.pivot_table(index=key,columns='grad_partition',values='grad_norm_per_token',aggfunc='mean').reset_index()
        if {'text_only','image_only'}.issubset(w.columns):
            eps=1e-12
            w['dom']=(w['image_only']-w['text_only'])/(w['image_only']+w['text_only']+eps)
            plt.figure(figsize=(10,4)); sns.histplot(w['dom'],bins=50,kde=True); plt.axvline(0,color='r',ls='--')
            plt.title('mixed batches dominance index'); plt.tight_layout(); plt.savefig(OUT/'fig_mixed_dom_hist.png',dpi=180); plt.show()

            dl=w.groupby('layer_depth',as_index=False)['dom'].mean()
            plt.figure(figsize=(11,4)); sns.lineplot(data=dl,x='layer_depth',y='dom',marker='o'); plt.axhline(0,color='r',ls='--')
            plt.title('mixed batches layer-wise dominance'); plt.tight_layout(); plt.savefig(OUT/'fig_mixed_dom_layer.png',dpi=180); plt.show()
    else:
        print('No text_and_vision rows.')
else:
    print('batch_mix_type not available, skip scenario A.')


## 4) 场景B：仅 text-img 数据（vision_only）优化分析

当没有text-only样本时，不再做“样本级模态冲突”解释，转为：
- token分区冲突（同一样本内 text token vs image token）
- 分层容量需求差异（SVD rank）

In [ ]:
v = meta[meta['batch_mix_type'].isin(['vision_only'])] if meta['batch_mix_type'].notna().any() else meta
if len(v)>0:
    # none梯度诊断
    if 'grad_was_none' in v.columns:
        nr=v.groupby(['layer_depth','grad_partition'],as_index=False)['grad_was_none'].mean()
        p=nr.pivot(index='layer_depth',columns='grad_partition',values='grad_was_none').sort_index()
        plt.figure(figsize=(10,8)); sns.heatmap(p,cmap='Blues')
        plt.title('none-grad ratio (vision-only focus)'); plt.tight_layout(); plt.savefig(OUT/'fig_none_ratio_vision_focus.png',dpi=200); plt.show()
else:
    print('No rows for scenario B')


## 5) 完整grad向量解析与缓存

In [ ]:
def to_npy(g):
    if g is None: return None
    if isinstance(g,(list,tuple,np.ndarray)):
        arr=np.asarray(g,dtype=np.float32).reshape(-1)
        h=hashlib.md5((str(arr.shape)+str(float(arr.sum()))).encode()).hexdigest()
        p=CACHE/f'{h}.npy'
        if not p.exists(): np.save(p,arr)
        return str(p)
    if isinstance(g,str):
        s=g.strip()
        if not s: return None
        if s.endswith('.npy') and Path(s).exists(): return s
        for fn in (json.loads, ast.literal_eval):
            try:
                x=fn(s)
                if isinstance(x,(list,tuple)):
                    arr=np.asarray(x,dtype=np.float32).reshape(-1)
                    h=hashlib.md5((s[:128]+str(len(arr))).encode()).hexdigest()
                    p=CACHE/f'{h}.npy'
                    if not p.exists(): np.save(p,arr)
                    return str(p)
            except: pass
    return None

vec = load_subset(LOG_PATH, usecols=['step','param_name','layer_depth','adapter_type','grad_partition','grad','batch_mix_type'], partition_in={'text_only','image_only','all'}, max_rows=MAX_VEC_ROWS)
if 'grad' not in vec.columns:
    print('No grad field; skip vector analyses')
else:
    vec['grad_path']=vec['grad'].apply(to_npy)
    vec=vec[vec['grad_path'].notna()].copy()
print('vector rows:',len(vec))


## 6) 方向冲突：cos(text,image)（按场景分层）

In [ ]:
def cos(a,b,eps=1e-12):
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+eps))

if len(vec)>0:
    sub=vec[vec['grad_partition'].isin(['text_only','image_only'])].copy()
    key=['step','param_name','layer_depth','adapter_type']
    if 'batch_mix_type' in sub.columns:
        key.append('batch_mix_type')
    t=sub[sub['grad_partition']=='text_only'][key+['grad_path']].rename(columns={'grad_path':'tp'})
    i=sub[sub['grad_partition']=='image_only'][key+['grad_path']].rename(columns={'grad_path':'ip'})
    pair=t.merge(i,on=key,how='inner').head(MAX_PAIR)

    vals=[]
    for _,r in pair.iterrows():
        a=np.load(r['tp']).reshape(-1); b=np.load(r['ip']).reshape(-1)
        m=min(len(a),len(b))
        vals.append(np.nan if m==0 else cos(a[:m],b[:m]))
    pair['cos']=vals
    pair=pair.dropna(subset=['cos'])

    plt.figure(figsize=(9,4)); sns.histplot(pair['cos'],bins=60,kde=True); plt.axvline(0,color='r',ls='--')
    plt.title('cos(text,image) overall'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_overall.png',dpi=180); plt.show()

    if 'batch_mix_type' in pair.columns:
        plt.figure(figsize=(10,4)); sns.boxplot(data=pair,x='batch_mix_type',y='cos')
        plt.title('cos(text,image) by batch composition'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_by_mix_type.png',dpi=180); plt.show()

    lc=pair.groupby('layer_depth',as_index=False)['cos'].mean()
    plt.figure(figsize=(11,4)); sns.lineplot(data=lc,x='layer_depth',y='cos',marker='o'); plt.axhline(0,color='r',ls='--')
    plt.title('layer-wise mean cos(text,image)'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_layer.png',dpi=180); plt.show()
else:
    print('No vectorized gradients available')


## 7) 子空间容量（SVD rank95）按场景/分区

In [ ]:
def rank_energy(X, e=0.95):
    if X.shape[0]<2: return np.nan
    X=X-X.mean(0,keepdims=True)
    s=np.linalg.svd(X,full_matrices=False,compute_uv=False)
    en=np.cumsum(s*s)/(np.sum(s*s)+1e-12)
    return int(np.searchsorted(en,e)+1)

if len(vec)>0:
    rec=[]
    max_each=180
    group_cols=['layer_depth','grad_partition']
    if 'batch_mix_type' in vec.columns: group_cols=['batch_mix_type']+group_cols

    for gk,g in vec.groupby(group_cols):
        gg=g.head(max_each)
        if len(gg)<2: continue
        arr=[np.load(p).reshape(-1) for p in gg['grad_path'].tolist()]
        d=min(len(x) for x in arr)
        if d<=1: continue
        X=np.stack([x[:d] for x in arr])
        r95=rank_energy(X,0.95)
        rec.append((*((gk,) if not isinstance(gk,tuple) else gk), len(gg), d, r95))

    if len(rec)>0:
        cols=group_cols+['n','dim','rank95']
        rdf=pd.DataFrame(rec,columns=cols)
        display(rdf.head())
        rdf.to_csv(OUT/'table_rank95_by_mix_and_partition.csv',index=False,encoding='utf-8-sig')

        if 'batch_mix_type' in rdf.columns:
            plt.figure(figsize=(11,5))
            sns.lineplot(data=rdf,x='layer_depth',y='rank95',hue='grad_partition',style='batch_mix_type',markers=True,dashes=False)
        else:
            plt.figure(figsize=(11,5))
            sns.lineplot(data=rdf,x='layer_depth',y='rank95',hue='grad_partition',marker='o')
        plt.title('rank95 by layer/partition/composition')
        plt.tight_layout(); plt.savefig(OUT/'fig_rank95_composition.png',dpi=180); plt.show()


## 8) 方案级优化结论（对应你的问题）

### 当数据主要是 text-only + text-img 混合：
- 训练/分析要按 `batch_mix_type=text_and_vision` 单独统计，避免被纯文本batch稀释。
- 指标优先用 `grad_norm_per_token`，并额外报告 mixed-batch dominance。
- 结论重点：样本级模态混合是否放大层级竞争。

### 当数据只有 text-img：
- 核心从“样本类型冲突”转为“token分区冲突”（同一样本内 text token vs image token）。
- 强化向量余弦 + SVD rank95 两条证据链。
- 增加 `none-grad` 热力图判别结构不可达层。